# Transformer vs Mamba: full study runner

**Choose what this session runs** in the first code cell (`STAGES`). On Kaggle, run one or two
stages per session with *Save Version -> Save & Run All*, so it keeps running with the browser closed.

Runs the whole study on a Colab or Kaggle GPU, top to bottom. **Every stage is resumable**:
data, checkpoints and results live in `BASE` (Google Drive on Colab), finished runs are skipped,
and an interrupted training run continues from its last checkpoint. After a disconnect, rerun the
setup cells, then the stage you were on.

| Stage | What it produces | Rough T4 time |
| --- | --- | --- |
| 1 Data | 1.5B FineWeb-Edu tokens + 50 PG-19 books, 16K BPE | 1-2 h, once |
| 2 LR sweep (S) | best learning rate per architecture | ~4-6 h |
| 3 Scale S | Transformer, Mamba, Mamba-LTI x 3 seeds | ~6-10 h |
| 4 Scale M | Transformer and Mamba, 1 seed | ~8-12 h |
| 5 Synthetic | MQAR and selective-copying grids | ~1-3 h |
| 6 Evaluation | test perplexity, recall split, PG-19, efficiency, delta analysis | ~1 h |
| 7 Figures | `report/figures/*` and `results.zip` | minutes |

Split stages across team members' accounts: they all write to the same Drive folder if it is shared.

In [ ]:
# Pick the stages for this session: data, sweep, S, M, synthetic, eval, figures
STAGES = {'data', 'sweep', 'S', 'M', 'synthetic', 'eval', 'figures'}

import os, sys, subprocess, json, glob, shutil
from pathlib import Path

ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    BASE = Path('/kaggle/working/afml')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/afml')   # use a shared Drive folder for the whole team
REPO = Path('/content/afml-transformer-vs-state-space-models') if not ON_KAGGLE else Path('/kaggle/working/repo')
BRANCH = 'upgrade/full-study'
DATA = BASE / 'data_fineweb_16k'
RUNS = BASE / 'runs'
RESULTS = BASE / 'results'
for p in (BASE, RUNS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)

if ON_KAGGLE:  # Kaggle wipes /kaggle/working: reuse earlier sessions' outputs added via "Add Input"
    found = glob.glob('/kaggle/input/**/data_fineweb_16k/meta.json', recursive=True)
    if found:
        DATA = Path(found[0]).parent
    for src in glob.glob('/kaggle/input/**/afml/runs/*', recursive=True):
        dst = RUNS / Path(src).name
        if not dst.exists():
            shutil.copytree(src, dst)
    for src in glob.glob('/kaggle/input/**/afml/results', recursive=True):
        shutil.copytree(src, RESULTS, dirs_exist_ok=True)
    print('data:', DATA, '| runs restored:', len(list(RUNS.iterdir())))

def sh(cmd, cwd=None):
    """Run a shell command and stream its output into the notebook; stop on failure."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'command failed: {cmd}')

if not REPO.exists():
    sh(f'git clone --branch {BRANCH} https://github.com/Gagan-1718/afml-transformer-vs-state-space-models.git {REPO}', cwd='/')
else:
    sh(f'git fetch origin && git checkout {BRANCH} && git pull')
os.chdir(REPO)
sh('pip install -q -r requirements.txt')
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')

**Optional, recommended: the official Mamba CUDA kernel.** It makes Mamba training several times
faster and is needed for the fair efficiency benchmark. Installation compiles code and can take a
while; if it fails, everything still runs on the pure-PyTorch chunked scan (`scan="auto"` picks the
kernel when it is importable).

In [ ]:
try:
    sh('pip install -q causal-conv1d mamba-ssm --no-build-isolation')
except RuntimeError as e:
    print('mamba-ssm not installed; using the chunked PyTorch scan.', e)
sh('python -c "from afml.models.mamba import HAS_CUDA_SCAN; print(\'mamba-ssm kernel available:\', HAS_CUDA_SCAN)"')
sh('python -m pytest -q')   # includes the kernel-vs-reference test when the kernel is installed

## Stage 1: build the dataset (once)

In [ ]:
if 'data' in STAGES and not (DATA / 'pg19_test_offsets.npy').exists():
    if (DATA / 'meta.json').exists():
        sh(f'python -m afml.data.build --out {DATA} --pg19-only')
    else:
        sh(f'python -m afml.data.build --config configs/data.json --out {DATA}')
print(json.dumps(json.loads((DATA / 'meta.json').read_text())['splits'], indent=2))

## Stage 2: parameter-matched configs and the learning-rate sweep (scale S)

In [ ]:
sh('python scripts/make_configs.py')
if 'sweep' in STAGES:
    sh(f'python scripts/sweep_lr.py --scale S --archs transformer mamba --data {DATA} --runs {RUNS} --results {RESULTS}')
BEST_LR = json.loads((RESULTS / 'S' / 'best_lr.json').read_text())
BEST_LR

## Stage 3: main runs at scale S (3 seeds each)

In [ ]:
def train_run(scale, arch, seed, lr):
    out = RUNS / f'{scale}_{arch}_s{seed}'
    if (out / 'summary.json').exists():
        print('done:', out.name); return
    sh(f'python -m afml.train --config configs/{scale}_{arch}.json --data {DATA} --out {out} '
       f'--set train.seed={seed} --set train.lr={lr}')

if 'S' in STAGES:
    for seed in (0, 1, 2):
        for arch in ('transformer', 'mamba', 'mamba_lti'):
            train_run('S', arch, seed, BEST_LR[arch])

## Stage 4: scale M (1 seed each)

Learning rates shrink with width; we scale the S optimum by 0.6, roughly the ratio of the default
LRs in `scripts/make_configs.py`. If you have time, sweep M with `scripts/sweep_lr.py --scale M`.

In [ ]:
if 'M' in STAGES:
    for arch in ('transformer', 'mamba'):
        train_run('M', arch, 0, round(BEST_LR[arch] * 0.6, 6))

## Stage 5: synthetic recall tasks

In [ ]:
SYN = RESULTS / 'synthetic'
if 'synthetic' in STAGES:
  sh(f'python -m afml.eval.synthetic --task selective_copy --archs transformer mamba mamba_lti '
   f'--d-models 64 --seq-lens 64 128 256 512 --out {SYN}/selective_copy.csv')
  sh(f'python -m afml.eval.synthetic --task mqar --archs transformer mamba mamba_lti '
   f'--d-models 64 128 256 --seq-lens 64 128 256 512 --out {SYN}/mqar.csv')

## Stage 6: evaluation

In [ ]:
for scale in ('S', 'M') if 'eval' in STAGES else ():
    runs = ' '.join(sorted(glob.glob(f'{RUNS}/{scale}_*_s*')))
    if not runs:
        continue
    sh(f'python -m afml.eval.perplexity --runs {runs} --data {DATA} --split test --out {RESULTS}/{scale}/perplexity.csv')
    seed0 = ' '.join(sorted(glob.glob(f'{RUNS}/{scale}_*_s0')))
    sh(f'python -m afml.eval.long_context --runs {seed0} --data {DATA} --max-len 16384 --out {RESULTS}/{scale}/long_context.csv')

if 'eval' in STAGES:
  sh(f'python -m afml.eval.bench --configs configs/M_transformer.json configs/M_mamba.json '
   f'--lengths 512 1024 2048 4096 8192 16384 32768 --batches 1 8 --out {RESULTS}/bench/bench.csv')
  sh(f'python -m afml.eval.analysis --mamba {RUNS}/M_mamba_s0 --transformer {RUNS}/M_transformer_s0 '
   f'--data {DATA} --out {RESULTS}/analysis')

## Stage 7: figures and a results archive to send back

In [ ]:
sh(f'python scripts/collect_results.py --runs {RUNS} --results {RESULTS}')
if 'figures' in STAGES:
  sh(f'python scripts/make_figures.py --results {RESULTS} --out {RESULTS}/figures')
archive = shutil.make_archive(str(BASE / 'results'), 'zip', RESULTS)
print('Send this file:', archive)
from IPython.display import Image, display
for f in sorted((RESULTS / 'figures').glob('*.png')):
    display(Image(filename=str(f)))